In [2]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

c:\Users\alexb\miniconda3\envs\gnome_BERTopic\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Data
df = pd.read_csv("../../data/data_bertopic.csv")
docs = [doc.replace('\xa0', '') for doc in df["ADVICE"]]

df_stat = pd.read_csv("../../data/data_participants.csv")
df_stat = df_stat.query("cond == 'ADV'")
assert len(docs) == len(df_stat)

# Models
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
base_model = BERTopic.load("../../results/NLP/BERTopic_model", embedding_model=embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 936.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1138.02it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
TOPICS_OF_INTEREST = [2, 4]      # base-model topic ids to test
MIN_JACCARD = 0.3                          # flag weak matches for manual review
VAR = "pl2_understanding"

results_dir = "../../results/NLP/robustness_check"

def load(path):
    return BERTopic.load(path, embedding_model=embedding_model)

def topic_columns(model, docs):
    distr, _ = model.approximate_distribution(docs)   # computed once per model
    return distr

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

def keywords(model, t, k=5):
    return ", ".join(w for w, _ in model.get_topic(t)[:k])

base_model = load("../../results/NLP/BERTopic_model")
base_assign = np.array(base_model.topics_)
base_distr = topic_columns(base_model, docs)

rows = []

def record(run, target, model, distr, topic_id, jac):
    r, p = pearsonr(df_stat[VAR], distr[:, topic_id])
    rows.append({
        "run_name": run, "base_topic": target, "matched_topic": topic_id,
        "jaccard": jac, "n": int((np.array(model.topics_) == topic_id).sum()),
        "r": r, "p": p, "keywords": keywords(model, topic_id),
        "needs_review": jac < MIN_JACCARD,
    })

# base model
for t in TOPICS_OF_INTEREST:
    record("base", t, base_model, base_distr, t, 1.0)

# robustness models
robustness_models = sorted(
    f for f in os.listdir(results_dir)
    if os.path.isdir(os.path.join(results_dir, f))
)

for name in robustness_models:
    model = load(os.path.join(results_dir, name))
    distr = topic_columns(model, docs)
    assign = np.array(model.topics_)
    candidates = [t for t in set(assign) if t != -1]

    for t in TOPICS_OF_INTEREST:
        base_docs = set(np.where(base_assign == t)[0])
        scores = {c: jaccard(base_docs, set(np.where(assign == c)[0])) for c in candidates}
        best = max(scores, key=scores.get)
        record(name, t, model, distr, best, scores[best])

results = pd.DataFrame(rows)
results.to_csv("../../results/NLP/robustness_check/results_all_topics.csv", index=False)

# summary per topic: range of n and r across runs
summary = results.groupby("base_topic").agg(
    n_min=("n", "min"), n_max=("n", "max"),
    r_min=("r", "min"), r_max=("r", "max"),
    max_p=("p", "max"), weakest_match=("jaccard", "min"),
)
print(summary)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 909.83it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 822.17it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 999.91it/s, Materializing param=p

            n_min  n_max     r_min     r_max         max_p  weakest_match
base_topic                                                               
2             104    144  0.481482  0.606439  3.656155e-59       0.727273
4              47     88  0.348431  0.578624  6.423884e-30       0.618421
